# Online ingest demo

Drop image(s) into `data/incoming/` and run the cells below. Each image is
**captioned by Qwen3-VL** and **embedded by SigLIP**, then written into the file
DBs the retriever reads:

- `data/sampled_images/image_NNNNN.jpg` (copied into the dataset sequence)
- `data/qwen3_vl_metadata/<stem>.json` (structured caption)
- `data/siglip_image_emb/<stem>.npy` (SigLIP image vector)
- appended to `data/sampled_dataset_metadata.json`

All logic lives in the `fashion_retrieval` package — this notebook only drives it.

In [ ]:
# Load the ingestion pipeline once (Qwen3-VL + SigLIP loaded here, reused for every image).
from fashion_retrieval.indexer.ingest import Ingestor

ingestor = Ingestor()
print("models loaded - drop image(s) into data/incoming/ and run the next cell")

## Ingest everything in `data/incoming/`

In [ ]:
import json
from PIL import Image
from IPython.display import display

results = ingestor.ingest_folder("data/incoming")   # moves processed files to _ingested/
if not results:
    print("data/incoming/ is empty - drop image(s) there and re-run this cell")
for stem, path, meta in results:
    print(f"\n{stem}  <-  {path}")
    print(json.dumps(meta, indent=2)[:600])
    im = Image.open(path).convert("RGB"); im.thumbnail((256, 256)); display(im)
if results:
    print(f"\ningested {len(results)} image(s).")

## Rebuild the indexes and search (optional)

Ingestion writes the file DBs immediately, but the FAISS indexes are built by the
offline indexer. Rebuild (cheap - reuses the stored `.npy`) then query, and the new
images are searchable via the same late-fusion + rerank path.

In [ ]:
from fashion_retrieval.indexer.index import build_indexes
from fashion_retrieval.retriever.retrieve import FusionRetriever

build_indexes()                     # text.faiss + image.faiss + records.jsonl
retriever = FusionRetriever()
for hit in retriever.search("A red tie and a white shirt in a formal setting.", k=5):
    print(f"{hit['score']:.3f}  (text {hit['text_sim']:.2f} / img {hit['image_sim']:.2f})  "
          f"{hit['image_path']}  matched={hit['matched']}")